# Hammer Curl Analysis
**Fixes applied:**
1. Rep counted on full extension (descent), not on curl up
2. 'Increase ROM' only fires at top of curl, not every frame
3. Swinging detected via shoulder Y movement, not elbow angle
4. Both arms tracked — dominant side detected

**Added:**
- 📥 Video input cell — accepts mp4, trims to 4s if longer
- 📊 Telemetry single chart
- 📊 Multi-chart dashboard (elbow angle, L/R symmetry, shoulder stability, rep duration)

In [ ]:
!pip install mediapipe==0.10.20 opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 7.7 MB/s eta 0:00:00


In [ ]:
import mediapipe as mp
print(mp.__version__)
print('solutions' in dir(mp))
mp_pose = mp.solutions.pose
print('Pose Ready')

0.10.20
True
Pose Ready


In [ ]:
import kagglehub
path = kagglehub.dataset_download('hasyimabdillah/workoutfitness-video')
print('Path to dataset files:', path)
!ls "{path}/hammer curl/"

100%|██████████| 4.32G/4.32G [00:44<00:00, 104MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/hasyimabdillah/workoutfitness-video/versions/5
'hammer curl_10.mp4'  'hammer curl_17.mp4'  'hammer curl_5.MOV'
'hammer curl_11.mp4'  'hammer curl_18.mp4'  'hammer curl_6.MOV'
'hammer curl_12.mp4'  'hammer curl_19.mp4'  'hammer curl_7.MOV'
'hammer curl_13.mp4'  'hammer curl_1.MOV'   'hammer curl_8.mp4'
'hammer curl_14.mp4'  'hammer curl_2.MOV'   'hammer curl_9.mp4'
'hammer curl_15.mp4'  'hammer curl_3.MOV'
'hammer curl_16.mp4'  'hammer curl_4.MOV'


## 📥 Video Input
Type the filename from the `hammer curl` dataset folder below (mp4 only).
If the video is longer than 5 seconds, only the first 4 seconds will be used.

In [ ]:
import cv2
import os
from IPython.display import display, Video

# ── SET FILENAME HERE ────────────────────────────────────────────────────
# Run the dataset cell above first to see available files.
# Example: 'hammer curl_18.mp4'
FILENAME = 'hammer curl_18.mp4'   # ← change this to any file from the folder

# ── VALIDATE ─────────────────────────────────────────────────────────────
if not FILENAME.lower().endswith('.mp4'):
    raise ValueError(f'❌ Only .mp4 files accepted. Got: {FILENAME}')

raw_path = os.path.join(path, 'hammer curl', FILENAME)

if not os.path.exists(raw_path):
    raise FileNotFoundError(f'❌ File not found: {raw_path}\nCheck the filename and run the dataset cell above to list available files.')

# ── CHECK DURATION & TRIM IF > 5s ────────────────────────────────────────
cap          = cv2.VideoCapture(raw_path)
fps          = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration     = total_frames / fps
w            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f'📹 File: {FILENAME} | Duration: {duration:.2f}s | {fps:.1f}fps | {w}x{h}')

MAX_DURATION   = 4.0   # seconds to keep
TRIM_THRESHOLD = 5.0   # trim only if longer than this

if duration > TRIM_THRESHOLD:
    print(f'✂️  Video is {duration:.2f}s — trimming to first {MAX_DURATION}s...')
    video_path = '/content/hammer_curl_input.mp4'
    out_trim   = cv2.VideoWriter(video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    max_frames = int(MAX_DURATION * fps)
    frame_idx  = 0
    while cap.isOpened() and frame_idx < max_frames:
        ret, frame = cap.read()
        if not ret: break
        out_trim.write(frame)
        frame_idx += 1
    out_trim.release()
    print(f'✅ Trimmed video saved: {video_path}')
else:
    video_path = raw_path
    print(f'✅ Video within limit — using as-is.')

cap.release()

# ── PLAY THE VIDEO ───────────────────────────────────────────────────────
print('\n▶️  Playing input video:')
display(Video(video_path, embed=True, width=640))
print(f'\n📌 video_path = "{video_path}"  ← used by all cells below')

📹 File: hammer curl_18.mp4 | Duration: 3.10s | 30.0fps | 1080x608
✅ Video within limit — using as-is.

▶️  Playing input video:



📌 video_path = "/root/.cache/kagglehub/datasets/hasyimabdillah/workoutfitness-video/versions/5/hammer curl/hammer curl_18.mp4"  ← used by all cells below


## Live Preview

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
from google.colab.patches import cv2_imshow

mp_pose = mp.solutions.pose

# ── SETTINGS ──────────────────────────────────────────────────────────────
FLEXION_THRESHOLD   = 50    # elbow fully curled
EXTENSION_THRESHOLD = 150   # elbow fully extended
ROM_TARGET          = 100   # ideal range of motion
GRAPH_HISTORY       = 120
SWING_THRESHOLD     = 15    # max shoulder Y movement (pixels) before swinging alert

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

def draw_neon_line(frame, p1, p2, color):
    overlay = frame.copy()
    cv2.line(overlay, p1, p2, color, 12, cv2.LINE_AA)
    cv2.addWeighted(overlay, 0.2, frame, 0.8, 0, frame)
    cv2.line(frame, p1, p2, color, 3, cv2.LINE_AA)

def draw_neon_joint(frame, point, color):
    x, y = point
    overlay = frame.copy()
    cv2.circle(overlay, (x, y), 15, color, -1)
    cv2.addWeighted(overlay, 0.3, frame, 0.7, 0, frame)
    cv2.circle(frame, (x, y), 6, (255,255,255), -1)

def draw_graph(frame, history):
    gh, gw, x0 = 150, 300, 20
    y0 = frame.shape[0] - gh - 20
    cv2.rectangle(frame, (x0,y0), (x0+gw, y0+gh), (20,20,20), -1)
    if len(history) > 1:
        for i in range(1, len(history)):
            x1 = x0 + int((i-1)*gw/GRAPH_HISTORY)
            y1 = y0 + gh - int(history[i-1]/180*gh)
            x2 = x0 + int(i*gw/GRAPH_HISTORY)
            y2 = y0 + gh - int(history[i]/180*gh)
            cv2.line(frame, (x1,y1), (x2,y2), (0,255,255), 2)

# ── VIDEO ─────────────────────────────────────────────────────────────────
cap   = cv2.VideoCapture(video_path)   # ← uses uploaded video
pose  = mp_pose.Pose()

rep_count    = 0
stage        = 'down'
angle_history = deque(maxlen=GRAPH_HISTORY)
min_angle, max_angle = 999, 0
prev_shoulder_y = None

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb)

    if results.pose_landmarks:
        lm      = results.pose_landmarks.landmark
        h, w, _ = frame.shape
        def pt(id): return (int(lm[id].x*w), int(lm[id].y*h))
        def pt_arr(id): return np.array([lm[id].x*w, lm[id].y*h])

        # Use whichever arm is more visible (higher visibility score)
        use_left = lm[11].visibility > lm[12].visibility
        if use_left:
            shoulder, elbow, wrist = pt(11), pt(13), pt(15)
        else:
            shoulder, elbow, wrist = pt(12), pt(14), pt(16)

        angle = calculate_angle(shoulder, elbow, wrist)
        angle_history.append(angle)

        min_angle = min(min_angle, angle)
        max_angle = max(max_angle, angle)
        rom = max_angle - min_angle

        # FIX 1 — count rep on full extension (descent)
        if angle < FLEXION_THRESHOLD and stage == 'down':
            stage = 'up'

        if angle > EXTENSION_THRESHOLD and stage == 'up':
            stage = 'down'
            rep_count += 1
            min_angle, max_angle = 999, 0

        # FIX 3 — swinging: detect shoulder Y movement
        shoulder_y = shoulder[1]
        swing_detected = False
        if prev_shoulder_y is not None:
            if abs(shoulder_y - prev_shoulder_y) > SWING_THRESHOLD:
                swing_detected = True
        prev_shoulder_y = shoulder_y

        draw_neon_line(frame, shoulder, elbow, (0,255,0))
        draw_neon_line(frame, elbow, wrist, (0,255,0))
        draw_neon_joint(frame, elbow, (0,255,255))

        cv2.putText(frame, f'Angle: {int(angle)}',  (elbow[0], elbow[1]-20), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
        cv2.putText(frame, f'Reps: {rep_count}',    (30,50),  cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 3)
        cv2.putText(frame, f'ROM: {int(rom)}',      (30,90),  cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,0), 2)

        warning = ''
        # FIX 2 — ROM alert only at top of curl
        if stage == 'up' and angle < FLEXION_THRESHOLD + 5:
            if rom < ROM_TARGET:
                warning = 'Increase Range of Motion'
        if swing_detected:
            warning = 'Avoid Swinging!'

        if warning:
            cv2.putText(frame, warning, (30,130), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 3)

        draw_graph(frame, angle_history)

    cv2_imshow(frame)
    if cv2.waitKey(1) & 0xFF == 27: break

cap.release()
cv2.destroyAllWindows()

## Save Processed Video

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque

mp_pose = mp.solutions.pose

FLEXION_THRESHOLD   = 50
EXTENSION_THRESHOLD = 150
ROM_TARGET          = 100
SWING_THRESHOLD     = 15

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

def draw_neon_line(frame, p1, p2, color):
    overlay = frame.copy()
    cv2.line(overlay, p1, p2, color, 12, cv2.LINE_AA)
    cv2.addWeighted(overlay, 0.2, frame, 0.8, 0, frame)
    cv2.line(frame, p1, p2, color, 3, cv2.LINE_AA)

def draw_neon_joint(frame, point, color):
    x, y = point
    overlay = frame.copy()
    cv2.circle(overlay, (x, y), 15, color, -1)
    cv2.addWeighted(overlay, 0.3, frame, 0.7, 0, frame)
    cv2.circle(frame, (x, y), 6, (255,255,255), -1)

cap    = cv2.VideoCapture(video_path)   # ← uses uploaded video
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

out  = cv2.VideoWriter('hammer_curl_ANALYZED.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
pose = mp_pose.Pose()

rep_count    = 0
stage        = 'down'
dominant_side = None
min_angle, max_angle = 999, 0
prev_shoulder_y = None

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(rgb)

    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        def pt(id): return (int(lm[id].x*width), int(lm[id].y*height))

        use_left = lm[11].visibility > lm[12].visibility
        if use_left:
            shoulder, elbow, wrist = pt(11), pt(13), pt(15)
            if dominant_side is None: dominant_side = 'Left'
        else:
            shoulder, elbow, wrist = pt(12), pt(14), pt(16)
            if dominant_side is None: dominant_side = 'Right'

        angle = calculate_angle(shoulder, elbow, wrist)
        min_angle = min(min_angle, angle)
        max_angle = max(max_angle, angle)
        rom = max_angle - min_angle

        if angle < FLEXION_THRESHOLD and stage == 'down': stage = 'up'
        if angle > EXTENSION_THRESHOLD and stage == 'up':
            stage = 'down'
            rep_count += 1
            min_angle, max_angle = 999, 0

        shoulder_y     = shoulder[1]
        swing_detected = prev_shoulder_y is not None and abs(shoulder_y - prev_shoulder_y) > SWING_THRESHOLD
        prev_shoulder_y = shoulder_y

        draw_neon_line(frame, shoulder, elbow, (0,255,0))
        draw_neon_line(frame, elbow, wrist, (0,255,0))
        draw_neon_joint(frame, elbow, (0,255,255))

        cv2.putText(frame, f'Reps: {rep_count}',          (30,50),  cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 3)
        cv2.putText(frame, f'Angle: {int(angle)}',        (30,90),  cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
        cv2.putText(frame, f'ROM: {int(rom)}',            (30,125), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,0), 2)
        cv2.putText(frame, f'Side: {dominant_side}',      (30,160), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)

        warning = ''
        if stage == 'up' and angle < FLEXION_THRESHOLD + 5 and rom < ROM_TARGET:
            warning = 'Increase Range of Motion'
        if swing_detected:
            warning = 'Avoid Swinging!'
        if warning:
            cv2.putText(frame, warning, (30,195), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 3)

    out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()
print('✅ Video saved as hammer_curl_ANALYZED.mp4')
!ls

from google.colab import files
files.download('hammer_curl_ANALYZED.mp4')

## Exercise Telemetry (Single Chart)

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

mp_pose = mp.solutions.pose

# ── SETTINGS ──────────────────────────────────────────────────────────────
FLEXION_THRESHOLD   = 80    # was 50 — loosened to match real curl depth
EXTENSION_THRESHOLD = 150
ROM_TARGET          = 100
SWING_THRESHOLD     = 15

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

cap    = cv2.VideoCapture(video_path)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

out  = cv2.VideoWriter('hammer_curl_ANALYZED.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
pose = mp_pose.Pose()

rep_count       = 0
stage           = 'down'
dominant_side   = 'Right'   # will be voted on first 30 frames
left_vis_votes  = 0
right_vis_votes = 0
frame_count     = 0
min_angle, max_angle = 999, 0
prev_shoulder_y = None
telem_data      = []
current_time    = 0.0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    current_time   = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
    rgb            = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results        = pose.process(rgb)

    avg_angle      = 0
    current_alerts = []

    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        def pt(id): return (int(lm[id].x*width), int(lm[id].y*height))
        def pt_arr(id): return np.array([lm[id].x*width, lm[id].y*height])

        l_shoulder = pt_arr(11); l_elbow = pt_arr(13); l_wrist = pt_arr(15)
        r_shoulder = pt_arr(12); r_elbow = pt_arr(14); r_wrist = pt_arr(16)

        l_angle = calculate_angle(l_shoulder, l_elbow, l_wrist)
        r_angle = calculate_angle(r_shoulder, r_elbow, r_wrist)
        avg_angle = (l_angle + r_angle) / 2

        # FIX 3 — dominant side voted over first 30 frames
        frame_count += 1
        if frame_count <= 30:
            if lm[11].visibility > lm[12].visibility:
                left_vis_votes += 1
            else:
                right_vis_votes += 1
        dominant_side = 'Left' if left_vis_votes >= right_vis_votes else 'Right'

        # ROM tracking
        min_angle = min(min_angle, avg_angle)
        max_angle = max(max_angle, avg_angle)
        rom = max_angle - min_angle

        # FIX 1+2 — rep counter uses dominant arm angle, threshold = 80
        dom_angle = l_angle if dominant_side == 'Left' else r_angle

        if dom_angle < FLEXION_THRESHOLD and stage == 'down': stage = 'up'
        if dom_angle > EXTENSION_THRESHOLD and stage == 'up':
            stage = 'down'
            rep_count += 1
            min_angle, max_angle = 999, 0

        # Shoulder swing detection
        shoulder_y     = shoulder_y = (l_shoulder[1] + r_shoulder[1]) / 2
        swing_detected = prev_shoulder_y is not None and abs(shoulder_y - prev_shoulder_y) > SWING_THRESHOLD
        prev_shoulder_y = shoulder_y

        # Alerts
        if stage == 'up' and dom_angle < FLEXION_THRESHOLD + 5 and rom < ROM_TARGET:
            current_alerts.append('Increase ROM!')
        if swing_detected:
            current_alerts.append('Avoid Swinging!')

        # Skeleton
        for p1, p2 in [(pt(11),pt(13)),(pt(13),pt(15)),(pt(12),pt(14)),(pt(14),pt(16))]:
            cv2.line(frame, p1, p2, (0,255,0), 2)
        cv2.putText(frame, f'Reps: {rep_count}',        (30,50),  cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 3)
        cv2.putText(frame, f'Angle: {int(dom_angle)}',  (30,90),  cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
        cv2.putText(frame, f'Side: {dominant_side}',    (30,125), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,0), 2)
        if current_alerts:
            cv2.putText(frame, current_alerts[0], (30,160), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 3)

    telem_data.append({
        'Time (s)':    round(current_time, 2),
        'Elbow Angle': round(avg_angle, 2),
        'Reps':        rep_count,
        'Stage':       stage,
        'Alerts':      ' | '.join(current_alerts) if current_alerts else None
    })
    out.write(frame)

if stage == 'up':
    rep_count += 1
    print(f'⚠️  Incomplete rep at video end — counted as Rep {rep_count}')

cap.release()
out.release()
cv2.destroyAllWindows()
print(f'✅ Video saved. | Total reps: {rep_count} | Dominant side: {dominant_side}')

# ── TELEMETRY PLOT ──
df = pd.DataFrame(telem_data)
df = df[df['Elbow Angle'] > 0]

fig = px.line(df, x='Time (s)', y='Elbow Angle', title='Hammer Curl Form Telemetry')
fig.add_hline(y=FLEXION_THRESHOLD,   line_dash='dash', line_color='cyan',   annotation_text='Curl Top')
fig.add_hline(y=EXTENSION_THRESHOLD, line_dash='dash', line_color='yellow', annotation_text='Curl Bottom')
fig.update_traces(hovertemplate='Time: %{x}s<br>Angle: %{y}°')

df_alerts = df.dropna(subset=['Alerts'])
if not df_alerts.empty:
    fig.add_trace(go.Scatter(
        x=df_alerts['Time (s)'], y=df_alerts['Elbow Angle'],
        mode='markers', marker=dict(color='red', size=8, symbol='x'),
        name='Form Alerts',
        hovertext=df_alerts['Alerts'], hoverinfo='text+x+y'
    ))
fig.show()

## Multi-Chart Biomechanics Dashboard

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

mp_pose = mp.solutions.pose

# ── SETTINGS ──────────────────────────────────────────────────────────────
FLEXION_THRESHOLD   = 80    # was 50 — loosened to match real curl depth
EXTENSION_THRESHOLD = 150
ROM_TARGET          = 100
SWING_THRESHOLD     = 15

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = abs(radians * 180.0 / np.pi)
    return 360 - angle if angle > 180 else angle

cap    = cv2.VideoCapture(video_path)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

out  = cv2.VideoWriter('hammer_curl_ANALYZED.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
pose = mp_pose.Pose()

rep_count       = 0
stage           = 'down'
dominant_side   = 'Right'
left_vis_votes  = 0
right_vis_votes = 0
frame_count     = 0
left_rom_min,  left_rom_max  = 999, 0
right_rom_min, right_rom_max = 999, 0
prev_shoulder_y = None
telem_data      = []
current_time    = 0.0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    current_time   = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
    rgb            = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results        = pose.process(rgb)

    l_angle = r_angle = avg_angle = dom_angle = 0
    shoulder_delta = 0
    current_alerts = []

    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        def pt(id): return (int(lm[id].x*width), int(lm[id].y*height))
        def pt_arr(id): return np.array([lm[id].x*width, lm[id].y*height])

        l_shoulder = pt_arr(11); l_elbow = pt_arr(13); l_wrist = pt_arr(15)
        r_shoulder = pt_arr(12); r_elbow = pt_arr(14); r_wrist = pt_arr(16)

        l_angle   = calculate_angle(l_shoulder, l_elbow, l_wrist)
        r_angle   = calculate_angle(r_shoulder, r_elbow, r_wrist)
        avg_angle = (l_angle + r_angle) / 2

        # FIX 3 — dominant side voted over first 30 frames
        frame_count += 1
        if frame_count <= 30:
            if lm[11].visibility > lm[12].visibility:
                left_vis_votes += 1
            else:
                right_vis_votes += 1
        dominant_side = 'Left' if left_vis_votes >= right_vis_votes else 'Right'

        # ROM per arm
        left_rom_min,  left_rom_max  = min(left_rom_min,  l_angle), max(left_rom_max,  l_angle)
        right_rom_min, right_rom_max = min(right_rom_min, r_angle), max(right_rom_max, r_angle)
        left_rom  = left_rom_max  - left_rom_min
        right_rom = right_rom_max - right_rom_min

        # FIX 1+2 — rep counter uses dominant arm angle, threshold = 80
        dom_angle = l_angle if dominant_side == 'Left' else r_angle

        if dom_angle < FLEXION_THRESHOLD and stage == 'down': stage = 'up'
        if dom_angle > EXTENSION_THRESHOLD and stage == 'up':
            stage = 'down'
            rep_count += 1
            left_rom_min,  left_rom_max  = 999, 0
            right_rom_min, right_rom_max = 999, 0

        # Shoulder stability
        shoulder_y = (l_shoulder[1] + r_shoulder[1]) / 2
        if prev_shoulder_y is not None:
            shoulder_delta = abs(shoulder_y - prev_shoulder_y)
            if shoulder_delta > SWING_THRESHOLD:
                current_alerts.append('Avoid Swinging!')
        prev_shoulder_y = shoulder_y

        # ROM alert at top of curl
        if stage == 'up' and dom_angle < FLEXION_THRESHOLD + 5:
            if left_rom < ROM_TARGET or right_rom < ROM_TARGET:
                current_alerts.append('Increase ROM!')

        # Skeleton both arms
        for p1, p2 in [(pt(11),pt(13)),(pt(13),pt(15)),(pt(12),pt(14)),(pt(14),pt(16))]:
            cv2.line(frame, tuple(np.array(p1).astype(int)), tuple(np.array(p2).astype(int)), (0,255,0), 2)

        cv2.putText(frame, f'Reps: {rep_count}',           (30,50),  cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 3)
        cv2.putText(frame, f'Dom Angle: {int(dom_angle)}', (30,90),  cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
        cv2.putText(frame, f'L: {int(l_angle)}  R: {int(r_angle)}', (30,125), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
        cv2.putText(frame, f'Side: {dominant_side}',       (30,160), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,0), 2)
        if current_alerts:
            cv2.putText(frame, current_alerts[0], (30,195), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,255), 3)

    telem_data.append({
        'Time (s)':       round(current_time, 2),
        'Avg Angle':      round(avg_angle, 2),
        'Dom Angle':      round(dom_angle, 2),
        'Left Angle':     round(l_angle, 2),
        'Right Angle':    round(r_angle, 2),
        'Shoulder Delta': round(shoulder_delta, 2),
        'Reps':           rep_count,
        'Stage':          stage,
        'Alerts':         ' | '.join(current_alerts) if current_alerts else None
    })
    out.write(frame)

if stage == 'up':
    rep_count += 1
    print(f'⚠️  Incomplete rep at video end — counted as Rep {rep_count}')

cap.release()
out.release()
cv2.destroyAllWindows()
print(f'✅ Video saved. | Total reps: {rep_count} | Dominant side: {dominant_side}')

# ── DASHBOARD ─────────────────────────────────────────────────────────────
df = pd.DataFrame(telem_data)
df = df[df['Avg Angle'] > 0]

rep_durations = df.groupby('Reps')['Time (s)'].agg(['min','max'])
rep_durations['Duration'] = rep_durations['max'] - rep_durations['min']
rep_durations = rep_durations[rep_durations.index > 0]

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.08,
    subplot_titles=(
        '1. Elbow Angle & Form Alerts',
        '2. Left vs Right Arm Symmetry',
        '3. Shoulder Stability  (lower = less swinging)',
        '4. Rep Duration & Fatigue Analysis'
    ),
    row_heights=[0.35, 0.25, 0.2, 0.2]
)

# ── Plot 1: Dominant Arm Elbow Angle ──
fig.add_trace(go.Scatter(
    x=df['Time (s)'], y=df['Dom Angle'],
    name='Dom Arm Angle', line=dict(color='royalblue', width=2)
), row=1, col=1)
fig.add_hline(y=FLEXION_THRESHOLD,   line_dash='dash', line_color='cyan',
              annotation_text='Curl Top',    row=1, col=1)
fig.add_hline(y=EXTENSION_THRESHOLD, line_dash='dash', line_color='yellow',
              annotation_text='Curl Bottom', row=1, col=1)

# Rep shading
rep_colors = ['rgba(200,200,200,0.15)', 'rgba(255,255,255,0)']
for i, rep in enumerate(rep_durations.index):
    fig.add_vrect(
        x0=rep_durations.loc[rep,'min'], x1=rep_durations.loc[rep,'max'],
        fillcolor=rep_colors[i%2], layer='below', line_width=0,
        annotation_text=f'Rep {rep}', annotation_position='top left',
        row=1, col=1
    )

# Alerts on chart 1
df_alerts = df.dropna(subset=['Alerts'])
fig.add_trace(go.Scatter(
    x=df_alerts['Time (s)'], y=df_alerts['Dom Angle'],
    mode='markers', marker=dict(color='red', size=8, symbol='x'),
    name='Alert Triggered',
    hovertext=df_alerts['Alerts'], hoverinfo='text+x+y'
), row=1, col=1)

# ── Plot 2: L vs R symmetry ──
fig.add_trace(go.Scatter(
    x=df['Time (s)'], y=df['Left Angle'],
    name='Left Arm', line=dict(color='teal', width=2)
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=df['Time (s)'], y=df['Right Angle'],
    name='Right Arm', line=dict(color='mediumpurple', width=2)
), row=2, col=1)

# ── Plot 3: Shoulder stability ──
fig.add_trace(go.Scatter(
    x=df['Time (s)'], y=df['Shoulder Delta'],
    name='Shoulder Movement', line=dict(color='darkorange', width=2)
), row=3, col=1)
fig.add_hline(y=SWING_THRESHOLD, line_dash='dash', line_color='red',
              annotation_text='Swing Threshold', row=3, col=1)

# ── Plot 4: Rep Durations ──
fig.add_trace(go.Bar(
    x=[f'Rep {r}' for r in rep_durations.index],
    y=rep_durations['Duration'],
    name='Time per Rep', marker_color='indianred'
), row=4, col=1)

fig.update_layout(
    title_text='Hammer Curl Biomechanics & Performance Dashboard',
    height=1200, hovermode='x unified', showlegend=True, template='plotly_dark'
)
fig.update_yaxes(title_text='Degrees (°)',  row=1, col=1)
fig.update_yaxes(title_text='Degrees (°)',  row=2, col=1)
fig.update_yaxes(title_text='Pixel Delta',  row=3, col=1)
fig.update_yaxes(title_text='Seconds',      row=4, col=1)
fig.update_xaxes(title_text='Time in Video (s)', row=4, col=1)

fig.show()